In [1]:
# !pip install langchain-google-genai langchain-core langchain-community pypdf

In [2]:
from langchain_community.document_loaders import PyPDFLoader

In [3]:
ruta = "./Montaje_de_un_Motor_Eléctrico_Básico.pdf"

In [4]:
loader = PyPDFLoader(ruta)

In [5]:
pages = loader.load()

In [6]:
for i, pages in enumerate(pages):
  print(f"---- Pagina {i+1} -----")
  print(f"---- Contenido {pages.page_content} -----")

---- Pagina 1 -----
---- Contenido 1. Materiales Necesarios: 
Para este proyecto, necesitarás los siguientes materiales: 
Componentes: 
• Estator: Bobina de cobre esmaltado, 1 m de longitud. 
• Rotor: Eje de acero (con diámetro de 5 mm y largo de 50 mm). 
• Imanes permanentes: 2 imanes de neodimio (dimensiones: 25 mm x 5 mm). 
• Escobillas de carbón: 2 piezas. 
• Carcasa del motor: 1 carcasa metálica con rosca para montaje (material de 
aluminio). 
• Colector: Commutador de cobre (típico para motores de corriente continua). 
• Soportes para el rotor: 2 cojinetes de acero. 
• Cables eléctricos: 2 cables de 1 metro cada uno (uno positivo y uno negativo). 
• Tornillos y tuercas: Para fijar las piezas entre sí. 
Herramientas: 
• Destornilladores (Phillips y plano). 
• Pistola de soldar (o soldador eléctrico). 
• Multímetro (para medición de corriente y resistencia). 
• Cinta aislante. 
• Prensa para ensamblaje de rodamientos. 
• Alicates. 
 
2. Preparación: 
2.1. Preparación de las piezas:

In [7]:
# !pip install langchain-text-splitters

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)

In [10]:
pages = loader.load()

In [11]:
chunks = text_splitter.split_documents(pages)

In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

In [13]:
MODEL = "gemini-2.5-flash-lite"
API_KEY = userdata.get('GEMINI_API_KEY')

In [14]:
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.5, google_api_key=API_KEY)

In [15]:
# resumenes = []

# for chunk in chunks[:2]:
#   response = llm.invoke(f"Haz un resumen de los puntos mas importantes del chunk: {chunk.page_content}")
#   resumenes.append(response.content)

# BBDD Vectorial

In [16]:
# !pip install langchain-chroma

In [17]:
from langchain_chroma import Chroma

In [20]:
!pip install langchain-huggingface

In [21]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

In [22]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [41]:
ruta = "./Contrato_Laboral_Individual_I.pdf"

In [42]:
loader = PyPDFLoader(ruta)

In [43]:
pages = loader.load()

In [45]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

In [46]:
docs_split = text_splitter.split_documents(pages)

In [47]:
print(f"Se han creado: {len(docs_split)} chunks de texto")

Se han creado: 11 chunks de texto


In [48]:
vectorstore = Chroma.from_documents(
    docs_split,
    embedding=embeddings,
    persist_directory="../data/chroma_db"
)

In [49]:
consulta = "Cual es el bonus de permanencia de CARLOS EDUARDO NAVARRO IBÁÑEZ"

In [50]:
resultados = vectorstore.similarity_search(consulta, 4)

In [53]:
print("Top documentos mas similares a la consulta:")
for i, doc in enumerate(resultados, start=1):
  print(f"------Contenido------{i}:{doc.page_content}")
  print(f"------Metadatos------{i}:{doc.metadata}")
  i=i+1

Top documentos mas similares a la consulta:
------Contenido------1:IV. FIRMA
Firmado en Barcelona, a 20 de marzo de 2026, por duplicado ejemplar .
POR LA EMPRESA EL DIRECTIVO
Elena Sofía Quintana ParedesCarlos Eduardo Navarro Ibáñez
Presidenta del Consejo DNI 78123649-K
Documento elaborado con datos totalmente ficticios y con fines demostrativos.
3
------Metadatos------1:{'title': 'Contrato Laboral Individual Iii', 'page_label': '3', 'page': 2, 'creator': 'ChatGPT', 'source': './Contrato_Laboral_Individual_I.pdf', 'total_pages': 3, 'producer': 'WeasyPrint 65.1', 'author': 'ChatGPT Canvas', 'creationdate': ''}
------Contenido------2:las necesidades del cargo.
QUINTA. Retribución
El  empleador  pagará  al  trabajador  Daniel  Alejandro  Ríos  Martínez las  siguientes  percepciones
económicas: Pago mediante transferencia a la cuenta ES91 0049 8822 3311 9900 7766.
SEXTA. Bonus de permanencia
Se establece un  bonus de permanencia de 25.000 € por cada periodo completo de 24 meses de
vigencia

In [54]:
response = llm.invoke(f"Cual es el bonus de permanencia de CARLOS EDUARDO NAVARRO IBÁÑEZ: {resultados}")

In [55]:
response

AIMessage(content='Según el documento, el bonus de permanencia para Carlos Eduardo Navarro Ibáñez es de **25.000 € por cada periodo completo de 24 meses de vigencia continuada del contrato**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db657-080f-7a13-980b-78a2d13bbea8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1001, 'output_tokens': 43, 'total_tokens': 1044, 'input_token_details': {'cache_read': 0}})

In [56]:
response = llm.invoke(f"Cual es el bonus de permanencia de CARLOS EDUARDO NAVARRO IBÁÑEZ")

# Chunking Strategies & Embeddings

In [58]:
import urllib.request

In [60]:
url = "https://arxiv.org/pdf/2005.11401"

In [61]:
urllib.request.urlretrieve(url, "rag_paper.pdf")

('rag_paper.pdf', <http.client.HTTPMessage at 0x7e31ab088890>)

In [62]:
loader = PyPDFLoader("rag_paper.pdf")

In [63]:
pages = loader.load()

In [66]:
print(f"Paginas cargadas: {len(pages)}")
print(f"Caracteres en pagina 1: {len(pages[0].page_content)}")
print(f"\n-- Muestra de pagina 1--\n {pages[0].page_content}")

Paginas cargadas: 19
Caracteres en pagina 1: 2899

-- Muestra de pagina 1--
 Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University College London;⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-
stream NLP tasks. However, their ability to access and precisely manipulate knowl-
edge is still limited, and hence on knowledge-intensive tasks, their performance
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pre-
trained models with a differentiable access mechanism to explicit non-param

In [71]:
!pip install langchain-classic

In [73]:
from langchain_classic.schema import Document

In [75]:
full_text = "\n\n".join([p.page_content for p in pages])
full_doc = [Document(page_content=full_text, metadata={"source":"rag_papre.pdf"})]

In [79]:
print(f"Total caracteres en el documento: {len(full_text)}")
print(f"Total caracteres en el documento: {len(full_text.split())}")

Total caracteres en el documento: 69092
Total caracteres en el documento: 9883


Estrategia 1: CharacterTextSplitter — el más simple

In [80]:
from langchain_text_splitters import CharacterTextSplitter

In [81]:
splitter_char = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
)

In [82]:
chunks_char = splitter_char.split_documents(full_doc)

In [86]:
print(f"Chunks generados: {len(chunks_char)}")
print(f"Tamaño medio: {sum(len(c.page_content) for c in chunks_char)/len(chunks_char)}")
print(f"Chunk de ejemplo: {chunks_char[2].page_content}")

Chunks generados: 19
Tamaño medio: 3634.5263157894738
Chunk de ejemplo: byθ that generates a current token based on a context of the previousi− 1 tokensy1:i−1, the original
inputx and a retrieved passagez.
To train the retriever and generator end-to-end, we treat the retrieved document as a latent variable.
We propose two models that marginalize over the latent documents in different ways to produce a
distribution over generated text. In one approach, RAG-Sequence, the model uses the same document
to predict each target token. The second approach, RAG-Token, can predict each target token based
on a different document. In the following, we formally introduce both models and then describe the
pη andpθ components, as well as the training and decoding procedure.
2.1 Models
RAG-Sequence Model The RAG-Sequence model uses the same retrieved document to generate
the complete sequence. Technically, it treats the retrieved document as a single latent variable that
is marginalized to get the seq2

RecursiveCharacterTextSplitter

In [132]:
splitter_recursive = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 100,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [133]:
chunks_recursiva = splitter_recursive.split_documents(full_doc)

In [134]:
print(f"Chunks generados: {len(chunks_recursiva)}")
print(f"Tamaño medio: {sum(len(c.page_content) for c in chunks_recursiva)/len(chunks_recursiva)}")
print(f"Chunk de ejemplo: {chunks_recursiva[2].page_content}")

Chunks generados: 108
Tamaño medio: 699.574074074074
Chunk de ejemplo: pare two RAG formulations, one which conditions on the same retrieved passages
across the whole generated sequence, and another which can use different passages
per token. We ﬁne-tune and evaluate our models on a wide range of knowledge-
intensive NLP tasks and set the state of the art on three open domain QA tasks,
outperforming parametric seq2seq models and task-speciﬁc retrieve-and-extract
architectures. For language generation tasks, we ﬁnd that RAG models generate
more speciﬁc, diverse and factual language than a state-of-the-art parametric-only
seq2seq baseline.
1 Introduction
Pre-trained neural language models have been shown to learn a substantial amount of in-depth knowl-
edge from data [47]. They can do so without any access to an external memory, as a parameterized


SemanticChunker

In [109]:
!pip install langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 7.3 MB/s eta 0:00:00


In [112]:
from langchain_experimental.text_splitter import SemanticChunker

In [119]:
splitter_semantic = SemanticChunker(
    embeddings = embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95
)

In [120]:
chunk_semantic = splitter_semantic.split_documents(full_doc)

In [121]:
print(f"Chunks generados: {len(chunk_semantic)}")
print(f"Tamaño medio: {sum(len(c.page_content) for c in chunk_semantic)/len(chunk_semantic)}")
print(f"Chunk de ejemplo: {chunk_semantic[2].page_content}")

Chunks generados: 36
Tamaño medio: 1918.2222222222222
Chunk de ejemplo: URL https://link.springer.com/chapter/10.1007%
2F978-3-319-24027-5_20 .


In [124]:
chunk_semantic[0].page_content

'Retrieval-Augmented Generation for\nKnowledge-Intensive NLP Tasks\nPatrick Lewis†‡, Ethan Perez⋆,\nAleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,\nMike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†\n†Facebook AI Research;‡University College London;⋆New York University;\nplewis@fb.com\nAbstract\nLarge pre-trained language models have been shown to store factual knowledge\nin their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-\nstream NLP tasks. However, their ability to access and precisely manipulate knowl-\nedge is still limited, and hence on knowledge-intensive tasks, their performance\nlags behind task-speciﬁc architectures. Additionally, providing provenance for their\ndecisions and updating their world knowledge remain open research problems. Pre-\ntrained models with a differentiable access mechanism to explicit non-parametric\nmemory have so far been only investigated for extractiv

MarkdownHeaderTextSplitter

In [125]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [126]:
# Ejemplo con texto Markdown estructurado
markdown_doc = """
# Introducción a RAG

RAG combina recuperación de documentos con generación de texto.
Es especialmente útil para preguntas sobre documentos específicos.

## Componentes principales

Los componentes de un sistema RAG son el retriever y el generador.
El retriever busca documentos relevantes en una base vectorial.

### El Retriever

El retriever convierte la query en un vector y busca los más cercanos.
Usa métricas como cosine similarity o producto escalar.

### El Generador

El generador (LLM) recibe el contexto recuperado y genera la respuesta.
Modelos como GPT-4 o Claude son opciones comunes.

## Evaluación

Evaluar RAG requiere métricas especializadas como RAGAS.
Las métricas principales son faithfulness y answer relevancy.
"""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

In [129]:
splitter_md = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
chunks_md = splitter_md.split_text(markdown_doc)

In [130]:
print(f"Chunks generados: {len(chunks_md)}")
for i, chunk in enumerate(chunks_md):
    print(f"\n--- Chunk {i+1} ---")
    print(f"📌 Metadata: {chunk.metadata}")
    print(f"📝 Contenido: {chunk.page_content[:200]}")

Chunks generados: 5

--- Chunk 1 ---
📌 Metadata: {'Header 1': 'Introducción a RAG'}
📝 Contenido: RAG combina recuperación de documentos con generación de texto.
Es especialmente útil para preguntas sobre documentos específicos.

--- Chunk 2 ---
📌 Metadata: {'Header 1': 'Introducción a RAG', 'Header 2': 'Componentes principales'}
📝 Contenido: Los componentes de un sistema RAG son el retriever y el generador.
El retriever busca documentos relevantes en una base vectorial.

--- Chunk 3 ---
📌 Metadata: {'Header 1': 'Introducción a RAG', 'Header 2': 'Componentes principales', 'Header 3': 'El Retriever'}
📝 Contenido: El retriever convierte la query en un vector y busca los más cercanos.
Usa métricas como cosine similarity o producto escalar.

--- Chunk 4 ---
📌 Metadata: {'Header 1': 'Introducción a RAG', 'Header 2': 'Componentes principales', 'Header 3': 'El Generador'}
📝 Contenido: El generador (LLM) recibe el contexto recuperado y genera la respuesta.
Modelos como GPT-4 o Claude son opcion

In [135]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# Recopilar métricas
strategies = {
    "CharacterTextSplitter": chunks_char,
    "RecursiveCharacter\nTextSplitter": chunks_recursiva,
    "SemanticChunker": chunk_semantic,
}

stats = []
for name, chunks in strategies.items():
    sizes = [len(c.page_content) for c in chunks]
    stats.append({
        "Estrategia": name,
        "Nº chunks": len(chunks),
        "Tamaño medio": np.mean(sizes),
        "Tamaño mínimo": np.min(sizes),
        "Tamaño máximo": np.max(sizes),
        "Desviación estándar": np.std(sizes),
    })

df_stats = pd.DataFrame(stats).set_index("Estrategia")
print(df_stats.round(0).to_string())

                                  Nº chunks  Tamaño medio  Tamaño mínimo  Tamaño máximo  Desviación estándar
Estrategia                                                                                                  
CharacterTextSplitter                    19        3635.0           1138           4556                883.0
RecursiveCharacter\nTextSplitter        108         700.0            138            797                139.0
SemanticChunker                          36        1918.0             29          37698               6266.0


In [136]:
import os
import shutil

def build_vectorstore(chunks, embeddings, persist_dir):
    """Crea un Chroma vectorstore a partir de una lista de chunks."""
    if os.path.exists(persist_dir):
        shutil.rmtree(persist_dir)
    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_dir,
    )

# Usamos HuggingFace para no gastar créditos en el indexado
print("Indexando CharacterTextSplitter...")
db_char = build_vectorstore(chunks_char, embeddings, "./chroma_char")

print("Indexando RecursiveCharacterTextSplitter...")
db_recursive = build_vectorstore(chunks_recursiva, embeddings, "./chroma_recursive")

print("Indexando SemanticChunker...")
db_semantic = build_vectorstore(chunk_semantic, embeddings, "./chroma_semantic")

print("\n✅ Los 3 vectorstores están listos")

Indexando CharacterTextSplitter...
Indexando RecursiveCharacterTextSplitter...
Indexando SemanticChunker...

✅ Los 3 vectorstores están listos


In [137]:
def compare_retrieval(query, vectorstores_dict, k=3):
    """Compara los chunks recuperados por cada estrategia ante una misma query."""
    print(f"\n{'='*70}")
    print(f"🔍 QUERY: {query}")
    print(f"{'='*70}")

    for name, db in vectorstores_dict.items():
        docs = db.similarity_search(query, k=k)
        print(f"\n📚 [{name}] — {len(docs)} chunks recuperados")
        print("-" * 50)
        for i, doc in enumerate(docs):
            print(f"  Chunk {i+1} ({len(doc.page_content)} chars): {doc.page_content[:180].strip()}...")

vectorstores = {
    "CharacterTextSplitter": db_char,
    "RecursiveCharacterTextSplitter": db_recursive,
    "SemanticChunker": db_semantic,
}

# Probar con distintas queries
queries = [
    "How does RAG combine retrieval with generation?",
    "What are the limitations of standard language models?",
    "How is the retriever trained in RAG?",
]

for query in queries:
    compare_retrieval(query, vectorstores, k=2)


🔍 QUERY: How does RAG combine retrieval with generation?

📚 [CharacterTextSplitter] — 2 chunks recuperados
--------------------------------------------------
  Chunk 1 (3186 chars): Table 4: Human assessments for the Jeopardy
Question Generation Task.
Factuality Speciﬁcity
BART better 7.1% 16.8%
RAG better 42.7% 37.4%
Both good 11.7% 11.8%
Both poor 17.7% 6.9%...
  Chunk 2 (4205 chars): minimize the negative marginal log-likelihood of each target,∑
j− logp(yj|xj) using stochastic
gradient descent with Adam [28]. Updating the document encoder BERTd during training...

📚 [RecursiveCharacterTextSplitter] — 2 chunks recuperados
--------------------------------------------------
  Chunk 1 (707 chars): total ngrams generated by different models. Table 5 shows that RAG-Sequence’s generations are
more diverse than RAG-Token’s, and both are signiﬁcantly more diverse than BART withou...
  Chunk 2 (531 chars): and the RAG model would perform equivalently to BART. The collapse could be due to a l